# EX_05 — Vector stores y retrieval (ejercicios)

**Notebook de referencia:** `notebook/05_Vectorstores_Retrieval.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Chunking

Implementa un chunker trivial por **número de caracteres** con solapamiento (`chunk_size`, `chunk_overlap`). Aplícalo a un texto largo en una lista de strings.


In [1]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(text), step):
        chunks.append(text[i:i + chunk_size])
    return chunks

long_text = "word " * 500  
print("Cantidad de chunks:", len(chunk_text(long_text)))

Cantidad de chunks: 16


## Actividad 2 — Embeddings + FAISS

Embedde los chunks (puede ser `sentence_transformers`) y construye un índice `faiss.IndexFlatIP` o `IndexFlatL2`. Recupera los top-3 para una query.

*Hint:* L2-normalize vectors if you treat inner product as cosine similarity.


In [2]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
docs = ["El gato come pescado.", "El coche es rojo.", "Me gustan los felinos.", "Python es un lenguaje."]
doc_embs = model.encode(docs)

# Crear índice FAISS (L2)
dim = doc_embs.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(doc_embs)

query = "mascotas y gatos"
query_emb = model.encode([query])

distances, indices = index.search(query_emb, k=3)
print("Top 3 índices recuperados:", indices[0])

/opt/anaconda3/envs/IA_MUBA/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-05-25 18:10:53.759367: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/anaconda3/envs/IA_MUBA/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/

Top 3 índices recuperados: [0 2 1]


## Actividad 3 — Métrica manual

Para una query y tres documentos **artificiales** (uno relevante, dos ruido), muestra scores de similitud y verifica que el relevante queda primero.


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

q = model.encode(["¿Cómo resetear el router?"])
d1 = model.encode(["Para resetear el router, pulsa el botón trasero 10 segundos."]) # Relevante
d2 = model.encode(["La receta de la paella lleva arroz."]) # Ruido
d3 = model.encode(["El sol sale por el este."]) # Ruido

print("Similitud Relevante:", cosine_similarity(q, d1)[0][0])
print("Similitud Ruido 1:", cosine_similarity(q, d2)[0][0])
print("Similitud Ruido 2:", cosine_similarity(q, d3)[0][0])

Similitud Relevante: 0.8142402
Similitud Ruido 1: 0.35119063
Similitud Ruido 2: 0.4262333
